In [2]:
!hmmsearch --max --nonull2 profiles/PF30808.hmm sequences/A0A010YIN8.fa

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.4 (Aug 2023); http://hmmer.org/
# Copyright (C) 2023 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  profiles/PF30808.hmm
# target sequence database:        sequences/A0A010YIN8.fa
# Max sensitivity mode:            on [all heuristic filters off]
# null2 bias corrections:          off
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       DUF8746  [M=64]
Accession:   PF30808.1
Description: Domain of unknown function (DUF8746)
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence   Description
    ------- ------ -----    ------- ------ -----   ---- --  --------   -----------
    1.3e-15   43.8   0.0    4.3

In [1]:
import numpy as np
import pandas as pd
import pyhmmer
import math

In [2]:
transition_df = pd.read_csv('transition.csv', index_col=0)
emission_df = pd.read_csv('emission.csv', index_col=0)

# MAKE STOCHASTIC
trans_len = len(transition_df)
transition_df.iloc[trans_len - 1, trans_len - 1] = 1 # loops back to end state
emission_df.loc['$'] = 0 # added symbol $ for end state
emission_df.iloc[20, trans_len - 1] = 1

A = transition_df.to_numpy()
B = emission_df.to_numpy()

In [4]:
emission_df

,M0,I0,M1,I1,M2,I2,M3,I3,M4,I4,...,I60,M61,I61,M62,I62,M63,I63,M64,I64,M65
A,0.0,0.068141,0.093280,0.068141,0.110199,0.068146,0.055058,0.068141,0.065332,0.068141,...,0.068141,0.307906,0.068141,0.036929,0.068141,0.009510,0.068141,0.009179,0.068141,0.0
C,0.0,0.012007,0.005717,0.012007,0.005729,0.012007,0.004957,0.012007,0.005094,0.012007,...,0.012007,0.004175,0.012007,0.004118,0.012007,0.002599,0.012007,0.000998,0.012007,0.0
D,0.0,0.062338,0.066717,0.062338,0.044202,0.062335,0.002106,0.062338,0.028498,0.062338,...,0.062338,0.037972,0.062338,0.050841,0.062338,0.001466,0.062338,0.023728,0.062338,0.0
E,0.0,0.065139,0.074059,0.065139,0.074407,0.065136,0.003331,0.065139,0.050838,0.065139,...,0.065139,0.057643,0.065139,0.035073,0.065139,0.002286,0.065139,0.853747,0.065139,0.0
F,0.0,0.031319,0.006805,0.031319,0.036970,0.031317,0.008204,0.031319,0.015482,0.031319,...,0.031319,0.008567,0.031319,0.005336,0.031319,0.016524,0.031319,0.001355,0.031319,0.0
G,0.0,0.090254,0.076815,0.090254,0.056293,0.090259,0.003967,0.090254,0.022231,0.090254,...,0.090254,0.117972,0.090254,0.028650,0.090254,0.002930,0.090254,0.008894,0.090254,0.0
H,0.0,0.024115,0.021318,0.024115,0.032837,0.024113,0.001705,0.024115,0.022389,0.024115,...,0.024115,0.020919,0.024115,0.009820,0.024115,0.001565,0.024115,0.003865,0.024115,0.0
I,0.0,0.037122,0.011386,0.037122,0.017471,0.037120,0.103891,0.037122,0.022440,0.037122,...,0.037122,0.014969,0.037122,0.009145,0.037122,0.036634,0.037122,0.001774,0.037122,0.0
K,0.0,0.068741,0.063037,0.068741,0.060860,0.068738,0.003593,0.068741,0.076634,0.068741,...,0.068741,0.054292,0.068741,0.031654,0.068741,0.002803,0.068741,0.009019,0.068741,0.0
L,0.0,0.067640,0.029404,0.067640,0.026234,0.067637,0.039339,0.067640,0.111013,0.067640,...,0.067640,0.026447,0.067640,0.014170,0.067640,0.819124,0.067640,0.003191,0.067640,0.0


In [3]:
def observable_T(A, B, y):
    C = np.diag(B[y])
    return C @ A

symbols = list(emission_df.index)
n = A.shape[0]
s = B.shape[0]
T = {}
for i, s in enumerate(symbols):
    T_i = observable_T(A, B, i)
    T[s] = T_i

In [4]:
with pyhmmer.easel.SequenceFile("sequences/A0A010YIN8.fa", digital=False) as sf:
    sequences = list(sf)

# for seq in sequences:
seq = sequences[0]
x = np.array([[1] if i == 0 else [0] for i in range(len(A))], dtype=np.float128)
for char in seq.sequence:
    sub = T[char] @ x
    x = sub
x = T['$'] @ x

Log-odds Ratio

In [5]:
log2_model = np.log2(x.sum())
log2_null = len(seq.sequence) * np.log2(0.05)
log_odds = log2_model - log2_null
log_odds

np.longdouble('-20.605994235880919607')

Log-likelihood 

In [6]:
likelihood = -np.log(x.sum())
likelihood

np.longdouble('907.0112043263246621')

In [16]:
def dagger(A):
    return np.conj(A).T

def kraus_operators(A, B):
    n = A.shape[0]
    s = B.shape[0]
    T = {}
    for i in range(s):
        T_i = observable_T(A, B, i)
        for j in range(n):
            sqrt_T_i = np.zeros(T_i.shape)
            sqrt_T_i[:,j] += np.sqrt(T_i[:,j])
            T[j,i] = sqrt_T_i
    return T

n = A.shape[0]
s = B.shape[0]

T = kraus_operators(A,B)

sum_i_j = []
for i in range(n):
    sum_j = []
    for j in range(s):
        if j == 0:
            sum_j = dagger(T[i, j]) @ T[i, j]
        else:
            sum_j += dagger(T[i, j]) @ T[i, j]
    # print(sum_j)
    # print()
    if i == 0:
        sum_i_j = sum_j.copy()
    else:
        sum_i_j += sum_j

print("FINAL")
print(sum_i_j.sum())

FINAL
131.00025497373736
